plan:
generate noisy circle
compute VR (Ripser) and Alpha(GUDHI)
compute Delaunay Triangulation (scipy.spatial) to see underlying Alpha complex
use persim for diagram comparison (Wasserstein/Bottleneck distance)
compare persistence diagrams
create comparison table
draw conclusions

In [ ]:
import os
import tadasets
import gudhi
import numpy as np
import matplotlib.pyplot as plt
import time
from ripser import ripser
from persim import plot_diagrams
from scipy.spatial import Delaunay, Voronoi


# ── Data helpers ──────────────────────────────────────────────────────────────

def compute_comparison(shape, n_points, noise, max_dim=2):
    """Generate a point cloud and compute Rips + Alpha persistence."""
    if shape == 'circle':
        pts = tadasets.dsphere(n=n_points, d=1, r=1, noise=noise)
    elif shape == 'torus':
        pts = tadasets.torus(n=n_points, c=2, a=1, noise=noise)
    elif shape == 'sphere':
        pts = tadasets.dsphere(n=n_points, d=2, r=1, noise=noise)
    else:
        raise ValueError(f"Unknown shape: {shape!r}")

    t0 = time.time()
    result    = ripser(pts, maxdim=max_dim)
    rips_ms   = (time.time() - t0) * 1000
    dgms_rips = result['dgms']
    num_edges = result['num_edges']

    t0 = time.time()
    ac  = gudhi.AlphaComplex(points=pts)
    st  = ac.create_simplex_tree()
    st.compute_persistence()
    alpha_ms      = (time.time() - t0) * 1000
    num_simplices = st.num_simplices()
    dgms_alpha    = [
        np.array([[b, d] for dim, (b, d) in st.persistence()
                  if dim == i and d != float('inf')]
                 or np.empty((0, 2)))
        for i in range(max_dim + 1)
    ]

    return pts, dgms_rips, num_edges, rips_ms, dgms_alpha, num_simplices, alpha_ms, noise


def _filter_dgms(dgms, threshold):
    """Keep infinite bars always; drop finite bars with persistence ≤ threshold."""
    if threshold <= 0:
        return dgms
    out = []
    for dgm in dgms:
        if len(dgm) == 0:
            out.append(dgm)
        else:
            inf_mask = ~np.isfinite(dgm[:, 1])
            fin_mask = np.isfinite(dgm[:, 1]) & ((dgm[:, 1] - dgm[:, 0]) > threshold)
            out.append(dgm[inf_mask | fin_mask])
    return out


# ── Plot helpers ──────────────────────────────────────────────────────────────

def plot_barcode(dgms, ax, title="Barcode"):
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    y = 0
    for dim, dgm in enumerate(dgms):
        if len(dgm) == 0:
            continue
        for b, d in dgm:
            if np.isfinite(d):
                ax.plot([b, d], [y, y], color=colors[dim % len(colors)],
                        linewidth=1.5, solid_capstyle='butt')
                y += 1
    ax.set_title(title)
    ax.set_yticks([])
    ax.set_xlabel("Filtration value")


def plot_landscape(dgms, ax, title="Landscape", n_landscapes=3):
    colors = ['tab:blue', 'tab:orange', 'tab:green']
    plotted = False
    for dim, dgm in enumerate(dgms):
        if len(dgm) == 0:
            continue
        finite = dgm[np.isfinite(dgm[:, 1])]
        if len(finite) == 0:
            continue
        t_min, t_max = finite[:, 0].min(), finite[:, 1].max()
        ts    = np.linspace(t_min, t_max, 500)
        tents = np.maximum(0, np.minimum(ts[None, :] - finite[:, 0:1],
                                         finite[:, 1:2] - ts[None, :]))
        sorted_tents = np.sort(tents, axis=0)[::-1]
        for k in range(min(n_landscapes, len(sorted_tents))):
            ax.plot(ts, sorted_tents[k], color=colors[dim % len(colors)],
                    alpha=max(0.3, 0.9 - 0.25 * k), linewidth=1,
                    label=f'H{dim} λ{k+1}')
        plotted = True
    ax.set_title(title)
    ax.set_xlabel("Filtration value")
    if plotted:
        ax.legend(fontsize=6)


# ── Primary comparison function ───────────────────────────────────────────────

def plot_comparison(pts,
                    dgms_rips, num_edges, rips_ms,
                    dgms_alpha, num_simplices, alpha_ms,
                    shape, n_points, noise, threshold=0.0,
                    show_pd=False, show_bc=False, show_pl=False,
                    save=False):
    """
    Render the 2-row Rips / Alpha comparison for one config.

    Columns shown are controlled by show_pd, show_bc, show_pl.
    If all three are False, all three columns are shown (default-all convention).
    Set save=True to write a PNG to tda_outputs/.
    """
    dgms_rips_f  = _filter_dgms(dgms_rips,  threshold)
    dgms_alpha_f = _filter_dgms(dgms_alpha, threshold)

    col_types = [t for flag, t in [(show_pd, 'pd'), (show_bc, 'bc'), (show_pl, 'pl')] if flag]
    if not col_types:
        col_types = ['pd', 'bc', 'pl']

    n_diag = len(col_types)
    fig = plt.figure(figsize=(6 + 5 * n_diag, 9), layout='constrained')
    fig.suptitle(
        f"{shape.capitalize()}  |  n={n_points}  |  noise={noise}  |  thresh={threshold}",
        fontsize=12, fontweight='bold'
    )
    gs = fig.add_gridspec(2, 1 + n_diag)

    if pts.shape[1] == 2:
        ax_pc = fig.add_subplot(gs[:, 0])
        ax_pc.scatter(pts[:, 0], pts[:, 1], s=5)
        ax_pc.set_aspect('equal')
    else:
        ax_pc = fig.add_subplot(gs[:, 0], projection='3d')
        ax_pc.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2)
    ax_pc.set_title("Point Cloud")

    for col_offset, col_type in enumerate(col_types):
        c = 1 + col_offset

        if col_type == 'pd':
            ax_r = fig.add_subplot(gs[0, c])
            plot_diagrams(dgms_rips_f, show=False, ax=ax_r)
            ax_r.set_title(f"Rips Diagram\nedges: {num_edges}  |  {rips_ms:.1f} ms")

            ax_a = fig.add_subplot(gs[1, c])
            plot_diagrams(dgms_alpha_f, show=False, ax=ax_a)
            ax_a.set_title(f"Alpha Diagram\nsimplices: {num_simplices}  |  {alpha_ms:.1f} ms")

        elif col_type == 'bc':
            ax_r = fig.add_subplot(gs[0, c])
            plot_barcode(dgms_rips_f, ax_r, title="Rips Barcode")

            ax_a = fig.add_subplot(gs[1, c])
            plot_barcode(dgms_alpha_f, ax_a, title="Alpha Barcode")

        elif col_type == 'pl':
            ax_r = fig.add_subplot(gs[0, c])
            plot_landscape(dgms_rips_f, ax_r, title="Rips Landscape")

            ax_a = fig.add_subplot(gs[1, c])
            plot_landscape(dgms_alpha_f, ax_a, title="Alpha Landscape")

    if save:
        os.makedirs('tda_outputs', exist_ok=True)
        shown    = '+'.join(col_types)
        filename = f'tda_outputs/{shape}_n{n_points}_ns{noise}_{shown}_th{threshold}.png'
        plt.savefig(filename, dpi=150, bbox_inches='tight')

    plt.show()


def run_comparison(shape, n_points, noise, threshold=0.0,
                   show_pd=True, show_bc=False, show_pl=False, save=False):
    """Compute Rips + Alpha for one shape config and render the comparison figure."""
    pts, dgms_rips, num_edges, rips_ms, dgms_alpha, num_simplices, alpha_ms, _ = \
        compute_comparison(shape, n_points, noise)
    plot_comparison(
        pts, dgms_rips, num_edges, rips_ms, dgms_alpha, num_simplices, alpha_ms,
        shape=shape, n_points=n_points, noise=noise, threshold=threshold,
        show_pd=show_pd, show_bc=show_bc, show_pl=show_pl, save=save,
    )

In [ ]:
# ── Delaunay / Alpha geometric view ──────────────────────────────────────────
from matplotlib.path import Path as MplPath
from matplotlib.patches import Circle, PathPatch


def _circumcenter_r2(p0, p1, p2):
    """
    Circumcenter and squared circumradius of triangle (p0, p1, p2).
    Returns (None, None) for degenerate triangles.
    """
    a = p1 - p0
    b = p2 - p0
    D = 2.0 * (a[0]*b[1] - a[1]*b[0])
    if abs(D) < 1e-12:
        return None, None
    ux = (b[1]*(a @ a) - a[1]*(b @ b)) / D
    uy = (a[0]*(b @ b) - b[0]*(a @ a)) / D
    cc = p0 + np.array([ux, uy])
    return cc, float((cc[0]-p0[0])**2 + (cc[1]-p0[1])**2)


def _voronoi_finite_polygons_2d(vor, radius=None):
    """
    Extend infinite Voronoi ridges to finite polygons by projecting each open
    ray to a far point, then sort region vertices counter-clockwise.
    """
    new_regions = []
    new_verts   = vor.vertices.tolist()
    center      = vor.points.mean(axis=0)
    if radius is None:
        radius = (vor.points.max() - vor.points.min()) * 3

    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_idx in enumerate(vor.point_region):
        verts = vor.regions[region_idx]

        if all(v >= 0 for v in verts):
            new_regions.append(verts)
            continue

        new_region = [v for v in verts if v >= 0]
        for p2, v1, v2 in all_ridges[p1]:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                continue
            t = vor.points[p2] - vor.points[p1]
            t /= np.linalg.norm(t)
            n         = np.array([-t[1], t[0]])
            midpoint  = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, n)) * n
            far_point = vor.vertices[v2] + direction * radius
            new_verts.append(far_point.tolist())
            new_region.append(len(new_verts) - 1)

        vs     = np.array([new_verts[v] for v in new_region])
        c      = vs.mean(axis=0)
        angles = np.arctan2(vs[:, 1] - c[1], vs[:, 0] - c[0])
        new_regions.append(np.array(new_region)[np.argsort(angles)].tolist())

    return new_regions, np.array(new_verts)


def plot_delaunay_alpha(pts, alpha_value, shape='', noise=None,
                        seed=0, circumcenter=False):
    """
    Two-panel figure (2-D only).

    Panel 1 : Full Voronoi cells (color-coded, with cell borders) + Delaunay triangulation.
    Panel 2 : Ball (r=sqrt(α)) ∩ Voronoi cell per point, with the true boundary of that
              intersection: circle arc clipped to the cell + cell edge clipped to the ball.
              Faint Delaunay, faint Voronoi borders, filled alpha 2-simplices, alpha edges.
              circumcenter=True plots triangle circumcenters — green when R²≤α
              (triangle is in alpha complex), red otherwise.

    alpha_value : GUDHI units — α = r²  (squared circumradius), NOT r.
    """
    if pts.shape[1] != 2:
        raise ValueError("plot_delaunay_alpha only supports 2-D point clouds.")

    n_points = len(pts)

    tri = Delaunay(pts)
    vor            = Voronoi(pts)
    regions, verts = _voronoi_finite_polygons_2d(vor)

    rng         = np.random.default_rng(seed)
    hues        = rng.permutation(np.linspace(0, 1, n_points, endpoint=False))
    cell_colors = plt.cm.hsv(hues)
    cell_colors[:, :3] = cell_colors[:, :3] * 0.35 + 0.65

    ac = gudhi.AlphaComplex(points=pts)
    st = ac.create_simplex_tree()
    alpha_edges = [
        s for s, f in st.get_filtration()
        if len(s) == 2 and f <= alpha_value
    ]
    alpha_triangles = [
        s for s, f in st.get_filtration()
        if len(s) == 3 and f <= alpha_value
    ]

    r    = alpha_value ** 0.5
    pad  = (pts.max() - pts.min()) * 0.05
    xlim = (pts[:, 0].min() - pad, pts[:, 0].max() + pad)
    ylim = (pts[:, 1].min() - pad, pts[:, 1].max() + pad)

    _n_circ    = 64
    _theta     = np.linspace(0, 2 * np.pi, _n_circ, endpoint=False)
    _cos, _sin = np.cos(_theta), np.sin(_theta)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), layout='constrained')
    noise_str = f" | noise={noise}" if noise is not None else ""
    fig.suptitle(
        f"{shape.capitalize()} | n={n_points}{noise_str}"
        f" | α={alpha_value} (r²)  →  r≈{r:.3f}",
        fontsize=12, fontweight='bold'
    )

    def _cell_clip(ax, polygon):
        n_v   = len(polygon)
        codes = [MplPath.MOVETO] + [MplPath.LINETO]*(n_v-1) + [MplPath.CLOSEPOLY]
        patch = PathPatch(MplPath(np.vstack([polygon, polygon[0]]), codes), visible=False)
        ax.add_patch(patch)
        return patch

    def _ball_clip(ax, cx, cy):
        xy    = np.c_[cx + r*_cos, cy + r*_sin]
        codes = [MplPath.MOVETO] + [MplPath.LINETO]*(_n_circ-1) + [MplPath.CLOSEPOLY]
        patch = PathPatch(MplPath(np.vstack([xy, xy[0]]), codes), visible=False)
        ax.add_patch(patch)
        return patch

    def _fill_voronoi_full(ax, fill_alpha=0.45):
        for idx, region in enumerate(regions):
            ax.fill(*verts[region].T, color=cell_colors[idx], alpha=fill_alpha, zorder=0)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

    def _draw_voronoi_borders(ax, lw=0.7, border_alpha=0.6):
        for idx, region in enumerate(regions):
            closed = np.vstack([verts[region], verts[region][0]])
            ax.plot(closed[:, 0], closed[:, 1],
                    color=cell_colors[idx], linewidth=lw, alpha=border_alpha, zorder=1)

    def _fill_voronoi_clipped(ax, fill_alpha=0.45):
        for idx, region in enumerate(regions):
            polygon = verts[region]
            cx, cy  = pts[idx]
            color   = cell_colors[idx]

            cc = _cell_clip(ax, polygon)
            bc = _ball_clip(ax, cx, cy)

            disk = Circle((cx, cy), r, color=color, alpha=fill_alpha,
                          linewidth=0, zorder=0)
            ax.add_patch(disk)
            disk.set_clip_path(cc)

            arc = Circle((cx, cy), r, fill=False,
                         edgecolor=color, linewidth=1.1, alpha=0.9, zorder=2)
            ax.add_patch(arc)
            arc.set_clip_path(cc)

            closed = np.vstack([polygon, polygon[0]])
            (line,) = ax.plot(closed[:, 0], closed[:, 1],
                              color=color, linewidth=1.1, alpha=0.9, zorder=2)
            line.set_clip_path(bc)

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

    # ── Panel 1 ───────────────────────────────────────────────────────────────
    _fill_voronoi_full(ax1)
    _draw_voronoi_borders(ax1)
    ax1.triplot(pts[:, 0], pts[:, 1], tri.simplices,
                color='k', linewidth=0.6, zorder=2)
    ax1.scatter(pts[:, 0], pts[:, 1], s=12, color='k', zorder=3)
    ax1.set_title("Voronoi cells + Delaunay Triangulation")
    ax1.set_aspect('equal')

    # ── Panel 2 ───────────────────────────────────────────────────────────────
    _fill_voronoi_clipped(ax2)
    _draw_voronoi_borders(ax2, lw=0.5, border_alpha=0.25)
    ax2.triplot(pts[:, 0], pts[:, 1], tri.simplices,
                color='k', linewidth=0.5, alpha=0.4, zorder=3)
    for i, j, k in alpha_triangles:
        t = pts[[i, j, k]]
        ax2.fill(t[:, 0], t[:, 1], color='steelblue', alpha=0.25, zorder=4)
    for i, j in alpha_edges:
        ax2.plot([pts[i, 0], pts[j, 0]], [pts[i, 1], pts[j, 1]],
                 color='k', linewidth=2.0, zorder=5)

    if circumcenter:
        for simplex, filt in st.get_filtration():
            if len(simplex) != 3:
                continue
            i, j, k = simplex
            cc, _ = _circumcenter_r2(pts[i], pts[j], pts[k])
            if cc is None:
                continue
            if not (xlim[0] <= cc[0] <= xlim[1] and ylim[0] <= cc[1] <= ylim[1]):
                continue
            dot_color = 'limegreen' if filt <= alpha_value else 'tomato'
            ax2.scatter(cc[0], cc[1], color=dot_color, s=22, zorder=6,
                        edgecolors='k', linewidths=0.4)

    ax2.scatter(pts[:, 0], pts[:, 1], s=12, color='k', zorder=7)
    ax2.set_title(f"Alpha complex  (α ≤ {alpha_value})")
    ax2.set_aspect('equal')


def run_geometric(shape, n_points, noise, alpha_values,
                  circumcenter=False, seed=0):
    """Compute one point cloud and render plot_delaunay_alpha for each alpha in alpha_values."""
    pts, *_, noise_val = compute_comparison(shape, n_points, noise)
    for alpha in alpha_values:
        plot_delaunay_alpha(pts, alpha_value=alpha, shape=shape, noise=noise_val,
                            seed=seed, circumcenter=circumcenter)

In [ ]:
run_geometric('circle', n_points=50, noise=0.1,
              alpha_values=[0.001, 0.01, 0.05, 0.1, 0.3, 0.7],
              circumcenter=True)

In [ ]:
configs = [
    dict(shape='circle', n_points=100, noise=0.05),
    dict(shape='circle', n_points=200, noise=0.10),
    dict(shape='circle', n_points=200, noise=0.30, threshold=0.05),
    dict(shape='torus',  n_points=300, noise=0.05),
    dict(shape='torus',  n_points=500, noise=0.10),
]
for cfg in configs:
    run_comparison(**cfg, show_pd=True)

In [ ]:
# ── Diagram distance comparison ───────────────────────────────────────────────
from persim import bottleneck, wasserstein

def compare_diagrams(dgms_rips, dgms_alpha, shape='', noise=None):
    """
    Compare Rips vs Alpha persistence diagrams via bottleneck and Wasserstein-1
    distances for each homology dimension.

    Prints a distance table, renders both diagrams side by side with per-dim
    distances in the subtitle, and returns
    {dim: {'bottleneck': float, 'wasserstein': float}}.
    """
    n_dims  = min(len(dgms_rips), len(dgms_alpha))
    results = {}

    for dim in range(n_dims):
        d_r = dgms_rips[dim]
        d_a = dgms_alpha[dim]
        d_r_fin = d_r[np.isfinite(d_r[:, 1])] if len(d_r) else d_r
        d_a_fin = d_a[np.isfinite(d_a[:, 1])] if len(d_a) else d_a

        if len(d_r_fin) == 0 and len(d_a_fin) == 0:
            bn, ws = 0.0, 0.0
        else:
            bn = bottleneck(d_r_fin, d_a_fin)
            ws = wasserstein(d_r_fin, d_a_fin)
        results[dim] = {'bottleneck': bn, 'wasserstein': ws}

    parts = []
    if shape:
        parts.append(shape.capitalize())
    if noise is not None:
        parts.append(f"noise={noise}")
    label = " | ".join(parts)
    print(f"\nRips vs Alpha — {label}" if label else "\nRips vs Alpha")
    print(f"{'':4} {'bottleneck':>12} {'wasserstein':>14}")
    print("─" * 32)
    for dim, d in results.items():
        print(f"H{dim}   {d['bottleneck']:>12.6f} {d['wasserstein']:>14.6f}")

    fig, (ax_r, ax_a) = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')

    plot_diagrams(dgms_rips,  show=False, ax=ax_r)
    ax_r.set_title(f"Rips — {label}" if label else "Rips")

    plot_diagrams(dgms_alpha, show=False, ax=ax_a)
    ax_a.set_title(f"Alpha — {label}" if label else "Alpha")

    dist_lines = [
        f"H{dim}: bn={d['bottleneck']:.4f}  ws={d['wasserstein']:.4f}"
        for dim, d in results.items()
    ]
    fig.suptitle("  ·  ".join(dist_lines), fontsize=10)
    plt.show()

    return results


def run_distance_comparison(shape, n_points, noise):
    """Compute Rips + Alpha for one config and run compare_diagrams."""
    _, dgms_rips, *_, dgms_alpha, _, _, noise_val = compute_comparison(shape, n_points, noise)
    return compare_diagrams(dgms_rips, dgms_alpha, shape=shape, noise=noise_val)


# ── usage ─────────────────────────────────────────────────────────────────────
configs = [
    dict(shape='circle', n_points=200, noise=0.05),
    dict(shape='torus',  n_points=300, noise=0.05),
]
for cfg in configs:
    run_distance_comparison(**cfg)